In [0]:
storage_account_name = "vinodininlpstorage"
storage_account_key = "Add your key
"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
from functools import reduce


In [0]:
from pyspark.sql import Row

files = dbutils.fs.ls(
    "abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles"
)

rows = []

for f in files:

    if "-tech.json" in f.name:
        category = "Technology"
    elif "-business.json" in f.name:
        category = "Business"
    elif "-science.json" in f.name:
        category = "Science"
    elif "-health.json" in f.name:
        category = "Health"
    else:
        category = "Other"

    rows.append(Row(path=f.path, category=category))

file_df = spark.createDataFrame(rows)

display(file_df)

path,category
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T09:59:43.5824408Z-tech.json,Technology
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:00:12.4185059Z-tech.json,Technology
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:10:47.1700020Z-business.json,Business
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:12:11.3241652Z-business.json,Business
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:12:34.6281377Z-business.json,Business
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:12:38.4595130Z-business.json,Business
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:17:19.4710578Z-science.json,Science
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:17:20.8058801Z-science.json,Science
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:21:39.1097700Z-health.json,Health
abfss://raw-landing@vinodininlpstorage.dfs.core.windows.net/articles/2026-06-20T10:21:42.2999945Z-health.json,Health


In [0]:
from pyspark.sql import functions as F

raw_list = []

for row in file_df.collect():

    temp = (
        spark.read
             .option("multiline", "true")
             .json(row.path)
             .withColumn("category", F.lit(row.category))
    )

    raw_list.append(temp)

In [0]:
from functools import reduce

raw_df = reduce(
    lambda x, y: x.unionByName(y),
    raw_list
)

In [0]:
raw_df.printSchema()

root
 |-- articles: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- content: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- publishedAt: string (nullable = true)
 |    |    |-- source: struct (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- url: string (nullable = true)
 |    |    |-- urlToImage: string (nullable = true)
 |-- status: string (nullable = true)
 |-- totalResults: long (nullable = true)
 |-- category: string (nullable = false)



In [0]:
raw_articles = (
    raw_df
    .withColumn("article", F.explode("articles"))
    .select(
        F.col("article.url").alias("url"),
        "category"
    )
)

raw_articles = raw_articles.withColumn(
    "url_hash",
    F.md5("url")
)

In [0]:
silver_path = "abfss://silver@vinodininlpstorage.dfs.core.windows.net/articles"

silver_df = spark.read.json(silver_path)

silver_df = silver_df.withColumn(
    "url_hash",
    F.md5("url")
)

In [0]:
gold_df = (
    silver_df.join(
        raw_articles.select("url_hash", "category"),
        on="url_hash",
        how="left"
    )
)

In [0]:
raw_articles = raw_articles.withColumnRenamed(
    "category",
    "raw_category"
)

gold_df = silver_df.join(
    raw_articles.select("url_hash", "raw_category"),
    "url_hash",
    "left"
)

In [0]:
gold_df.groupBy("raw_category").count().show()

+------------+-----+
|raw_category|count|
+------------+-----+
|     Science| 1107|
|      Health| 1234|
|  Technology| 3178|
|    Business| 1574|
|        NULL|    2|
+------------+-----+



In [0]:
from pyspark.sql import functions as F

gold_df = gold_df.withColumn(
    "date",
    F.to_date("publishedAt")
)

Sentiment Trends


In [0]:
sentiment_trend = (
    gold_df
    .filter(F.col("raw_category").isNotNull())
    .groupBy("raw_category", "date")
    .agg(
        F.avg("sentiment_scores.positive")
        .alias("avg_sentiment_score")
    )
)

display(sentiment_trend)

raw_category,date,avg_sentiment_score
Health,2026-06-18,0.03415637860082301
Business,2026-06-21,0.08559748427672946
Technology,2026-06-19,0.1501470588235294
Science,2026-06-19,0.056775510204081676
Technology,2026-06-20,0.14602739726027406
Health,2026-06-19,0.08877647058823539
Business,2026-06-20,0.07954545454545447
Science,2026-06-18,0.005226480836236937
Business,2026-06-18,0.07255681818181807
Science,2026-06-20,0.07007999999999989


In [0]:
sentiment_trend.show(20, truncate=False)

+------------+----------+--------------------+
|raw_category|date      |avg_sentiment_score |
+------------+----------+--------------------+
|Health      |2026-06-18|0.03415637860082301 |
|Business    |2026-06-21|0.08559748427672946 |
|Technology  |2026-06-19|0.1501470588235294  |
|Science     |2026-06-19|0.056775510204081676|
|Technology  |2026-06-20|0.14602739726027406 |
|Health      |2026-06-19|0.08877647058823539 |
|Business    |2026-06-20|0.07954545454545447 |
|Science     |2026-06-18|0.005226480836236937|
|Business    |2026-06-18|0.07255681818181807 |
|Science     |2026-06-20|0.07007999999999989 |
|Technology  |2026-06-18|0.21381316998468677 |
|Business    |2026-06-19|0.06182773109243695 |
|Health      |2026-06-20|0.06796116504854366 |
|Business    |2026-06-22|0.0                 |
|Technology  |2026-06-21|0.2014238410596027  |
|Technology  |2026-06-22|0.11                |
|Health      |2026-06-21|0.13106666666666664 |
|Health      |2026-06-22|0.1                 |
|Science     

Top Entities


In [0]:
top_entities = (
    gold_df
    .filter(F.col("raw_category").isNotNull())
    .withColumn("entity", F.explode("entities"))
    .withColumn("week", F.weekofyear("date"))
    .groupBy("raw_category", "week", "entity")
    .count()
    .orderBy("raw_category", "week", F.desc("count"))
)

display(top_entities)

raw_category,week,entity,count
Business,25,AI:Skill,326
Business,25,U.S.:Location,189
Business,25,Yahoo Finance:Organization,135
Business,25,SpaceX:Organization,134
Business,25,US:Location,129
Business,25,Iran:Location,124
Business,25,Fox Business:Organization,120
Business,25,Amazon:Organization,110
Business,25,Trump:Person,106
Business,25,ASML:Organization,101


In [0]:
gold_entities_path = "abfss://gold@vinodininlpstorage.dfs.core.windows.net/top_entities"

top_entities.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_entities_path)

In [0]:
display(
    spark.read.format("delta").load(gold_entities_path)
)

raw_category,week,entity,count
Business,25,AI:Skill,326
Business,25,U.S.:Location,189
Business,25,Yahoo Finance:Organization,135
Business,25,SpaceX:Organization,134
Business,25,US:Location,129
Business,25,Iran:Location,124
Business,25,Fox Business:Organization,120
Business,25,Amazon:Organization,110
Business,25,Trump:Person,106
Business,25,ASML:Organization,101



Trending Keywords


In [0]:
trending_keywords = (
    gold_df
    .withColumn("keyword", F.explode("keyphrases"))
    .filter(
        F.col("date") >= F.date_sub(F.current_date(), 7)
    )
    .groupBy("raw_category", "keyword")
    .count()
    .orderBy(F.desc("count"))
)

display(trending_keywords)

raw_category,keyword,count
Technology,July,111
Technology,AI,108
Science,researchers,106
Technology,Nintendo Life,103
Technology,game,102
Technology,Patch Notes,99
Technology,iOS,98
Technology,Players,92
Science,Space Daily,88
Technology,BuzzFeed,87


In [0]:
gold_sentiment_path = "abfss://gold@vinodininlpstorage.dfs.core.windows.net/sentiment_trends"

sentiment_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_sentiment_path)

In [0]:
gold_keywords_path = "abfss://gold@vinodininlpstorage.dfs.core.windows.net/trending_keywords"

trending_keywords.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_keywords_path)

In [0]:
gold_sentiment_path = "abfss://gold@vinodininlpstorage.dfs.core.windows.net/sentiment_trends"

sentiment_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "replaceWhere",
        "date >= '2026-06-20'"
    ) \
    .save(gold_sentiment_path)

In [0]:
gold_path = f"abfss://gold@vinodininlpstorage.dfs.core.windows.net/sentiment_trends"
df = spark.read.format("delta").load(gold_path)
df.show(5)

+------------+----------+--------------------+
|raw_category|      date| avg_sentiment_score|
+------------+----------+--------------------+
|      Health|2026-06-18|0.034156378600823004|
|    Business|2026-06-21| 0.08559748427672943|
|  Technology|2026-06-19| 0.15014705882352922|
|     Science|2026-06-19| 0.05654471544715452|
|    Business|2026-06-22|                 0.0|
+------------+----------+--------------------+
only showing top 5 rows
